# HW 07: Signal-Dependent Noise**Computational Sensorimotor Control — Week 7**Complete the code cells below by filling in the lines marked `### YOUR CODE ###`. Each answer requires at most 2 lines of code.

In [ ]:
!pip3 install git+https://github.com/tarkeshsingh/computational-sensorimotor-control.git#subdirectory=smc -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.figsize': (10, 5), 'font.size': 11,
                      'axes.grid': True, 'grid.alpha': 0.3})
from smc import (
    Arm, make_muscles, Q_REF, MUSCLE_DEFS,
    mass_matrix, coriolis, joint_accelerations,
)

arm = Arm(); ik = arm.inverse_kinematics; fk = arm.forward_kinematics; Jfn = arm.jacobian
start_hand = np.array(fk(Q_REF))

R_ARC = 0.06; T_ARC = 0.8; dt = 0.001; k_noise = 0.15
R_SH = np.array([m[3] for m in MUSCLE_DEFS])
R_EL = np.array([m[4] for m in MUSCLE_DEFS])
MUSCLE_NAMES = ['Pec', 'BicL', 'BicS', 'Delt', 'TriL', 'TriLg']
NAVY = '#1B2A4A'; TEAL = '#2E86AB'; RED = '#E74C3C'; GREEN = '#27AE60'; GRAY = '#7F8C8D'

# ── Helpers from Lab 07 ──
def minjerk_basis(t, T):
    tau = np.clip(t/T, 0, 1)
    return 10*tau**3 - 15*tau**4 + 6*tau**5

def minjerk_arc(R, T=0.8, dt=0.001):
    center = start_hand + np.array([R, 0])
    t = np.arange(0, T+dt/2, dt); s = minjerk_basis(t, T); th = np.pi*(1-s)
    return t, np.column_stack([center[0]+R*np.cos(th), center[1]+R*np.sin(th)])

def minjerk_reach(target, T=0.8, dt=0.001):
    t = np.arange(0, T+dt/2, dt); s = minjerk_basis(t, T)
    return t, start_hand[None,:] + s[:,None]*(target - start_hand)[None,:]

def dense_arc(R, n=500):
    center = start_hand + np.array([R, 0])
    th = np.linspace(np.pi, 0, n)
    return np.column_stack([center[0]+R*np.cos(th), center[1]+R*np.sin(th)])

def id_torques(t_arr, pos, dt):
    n = len(t_arr)
    q = np.array([ik(pos[i,0], pos[i,1]) for i in range(n)])
    qd = np.gradient(q, dt, axis=0); qdd = np.gradient(qd, dt, axis=0)
    tau = np.zeros((n,2))
    for i in range(n):
        M = mass_matrix(q[i]); C = coriolis(q[i], qd[i])
        tau[i] = M @ qdd[i] + C
    return q, qd, qdd, tau

def add_sdn(tau, k, rng):
    return tau + k * np.abs(tau) * rng.standard_normal(2)

def forward_sim_noisy(q0, qd0, tau_nom, dt, k, rng):
    n = len(tau_nom)
    q = np.zeros((n,2)); qd = np.zeros((n,2)); hand = np.zeros((n,2))
    q[0] = q0.copy(); qd[0] = qd0.copy(); hand[0] = fk(q0)
    for i in range(n-1):
        tau_noisy = add_sdn(tau_nom[i], k, rng)
        qdd = joint_accelerations(q[i], qd[i], tau_noisy)
        q[i+1] = q[i] + qd[i]*dt + 0.5*qdd*dt**2
        qd[i+1] = qd[i] + qdd*dt; hand[i+1] = fk(q[i+1])
    return hand

print("Setup complete.")

---## Q1: Noise Accumulation on the Arc (15 points)Run 200 noisy ID trials on the 6 cm semicircular arc (T = 800 ms). Examine how noise accumulates along the trajectory.

### Q1.1 (5 pts) — Plot 200 noisy hand paths on the arcRun the Monte Carlo and plot all 200 paths overlaid on the desired arc. Mark the start position.

In [ ]:
t_arc, pos_arc = minjerk_arc(R_ARC, T_ARC, dt)
q_arc, qd_arc, _, tau_arc = id_torques(t_arc, pos_arc, dt)
des = dense_arc(R_ARC)
N = 200

rng = np.random.default_rng(42)
all_hands = []  # store full hand trajectories
for trial in range(N):
    ### YOUR CODE ### (1 line: call forward_sim_noisy and append to all_hands)
    ...  # Replace this line

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(des[:,0]*100, des[:,1]*100, color=GRAY, lw=1, ls=':', label='Desired arc')
for h in all_hands:
    ax.plot(h[:,0]*100, h[:,1]*100, color=NAVY, alpha=0.04, lw=0.5)
ax.plot(pos_arc[:,0]*100, pos_arc[:,1]*100, color=TEAL, lw=2.5, label='Nominal ID', zorder=5)
ax.plot(pos_arc[0,0]*100, pos_arc[0,1]*100, 'D', color=NAVY, ms=9, mec='white', mew=1.5, zorder=10, label='Start')
ax.set_xlabel('x (cm)'); ax.set_ylabel('y (cm)'); ax.set_aspect('equal')
ax.set_title('200 noisy ID trials on the 6 cm arc'); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

### Q1.2 (5 pts) — Path deviation SD over timeCompute the SD of the hand position (distance from the nominal trajectory) at every timestep across the 200 trials. Plot SD(t) vs. time.

In [ ]:
# Stack all trajectories: shape (N, n_timesteps, 2)
hand_stack = np.array(all_hands)  # (200, n, 2)
nominal = pos_arc[None, :hand_stack.shape[1], :]  # (1, n, 2)

# Deviation of each trial from nominal at each timestep
deviations = np.linalg.norm(hand_stack - nominal, axis=2)  # (200, n)

### YOUR CODE ### (1 line: compute SD across trials at each timestep)
...  # Replace this line

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t_arc[:len(sd_t)]*1000, sd_t, color=NAVY, lw=2)
ax.set_xlabel('Time (ms)'); ax.set_ylabel('Path deviation SD (cm)')
ax.set_title('Noise accumulation along the arc trajectory')
plt.tight_layout(); plt.show()

print(f"Final endpoint SD: {sd_t[-1]:.3f} cm")
print(f"SD at mid-trajectory: {sd_t[len(sd_t)//2]:.3f} cm")

### Q1.3 (5 pts) — Where does the noise accumulate?At what percentage of the movement is 50% of the final endpoint SD reached? At what percentage is 90%? Explain why the second half of the arc accumulates more noise than the first half.

In [ ]:
final_sd = sd_t[-1]

### YOUR CODE ### (2 lines: find the timestep index where sd_t first exceeds 50% and 90% of final_sd)
...  # Replace this line

pct_50 = idx_50 / len(sd_t) * 100
pct_90 = idx_90 / len(sd_t) * 100
print(f"50% of final SD reached at {pct_50:.0f}% of the movement")
print(f"90% of final SD reached at {pct_90:.0f}% of the movement")
# Print your explanation below
print("Explanation: [Your explanation here]")

---## Q2: Direction Dependence of Endpoint Scatter (20 points)Run 200 noisy ID trials on center-out reaches in 4 directions (0°, 45°, 90°, 135°), each 10 cm at T = 800 ms.

### Q2.1 (8 pts) — Endpoint scatter in 4 directionsFill in the target computation for each direction and run the Monte Carlo.

In [ ]:
directions = [0, 45, 90, 135]  # degrees
D = 0.10  # 10 cm
T = 0.8; N = 200

eps_by_dir = {}
for deg in directions:
    ### YOUR CODE ### (1 line: compute target as start_hand + D * [cos(angle), sin(angle)])
    ...  # Replace this line

    t_r, pos_r = minjerk_reach(tgt, T, dt)
    q_r, qd_r, _, tau_r = id_torques(t_r, pos_r, dt)
    rng = np.random.default_rng(42)
    eps = np.zeros((N, 2))
    for trial in range(N):
        h = forward_sim_noisy(q_r[0], qd_r[0], tau_r, dt, k_noise, rng)
        eps[trial] = h[-1]
    eps_by_dir[deg] = (eps, tgt)
    sd = np.std(np.linalg.norm(eps - tgt, axis=1)) * 100
    print(f"  {deg:3d} deg: SD = {sd:.3f} cm, peak |tau| = {np.max(np.abs(tau_r)):.2f} N·m")

# Plot 2x2 grid
fig, axes = plt.subplots(2, 2, figsize=(10, 10))
for idx, deg in enumerate(directions):
    ax = axes[idx//2, idx%2]
    eps, tgt = eps_by_dir[deg]
    tx, ty = tgt[0]*100, tgt[1]*100
    ax.scatter(eps[:,0]*100, eps[:,1]*100, s=8, c=NAVY, alpha=0.4)
    ax.plot(tx, ty, 'k+', ms=12, mew=2, zorder=9)
    circ = plt.Circle((tx, ty), 0.3, fill=False, color='k', lw=1.5, zorder=8)
    ax.add_patch(circ)
    ax.plot(start_hand[0]*100, start_hand[1]*100, 'D', color=RED, ms=8, mec='white', mew=1.5, zorder=10)
    sd = np.std(np.linalg.norm(eps - tgt, axis=1)) * 100
    ax.set_title(f'{deg}° (SD = {sd:.3f} cm)', fontweight='bold')
    ax.set_xlabel('x (cm)'); ax.set_ylabel('y (cm)'); ax.set_aspect('equal')
plt.suptitle('Endpoint scatter in 4 reach directions', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

### Q2.2 (6 pts) — Why are the SDs different?In 2–3 sentences, explain why the endpoint SD varies across directions. Consider: which direction requires the largest torques? Why?**Your answer:***[Write your answer here]*

### Q2.3 (6 pts) — Jacobian prediction for the 0° reachCompute the Jacobian at the 0° target posture. Which column maps to more x-scatter? Does the simulation agree?

In [ ]:
tgt_0 = eps_by_dir[0][1]
q_tgt_0 = np.array(ik(tgt_0[0], tgt_0[1]))

### YOUR CODE ### (1 line: compute the Jacobian at the target posture)
...  # Replace this line

print("Jacobian at 0° target posture:")
print(f"  dhand/dq1 = ({J[0,0]:.3f}, {J[1,0]:.3f}) m/rad  (shoulder)")
print(f"  dhand/dq2 = ({J[0,1]:.3f}, {J[1,1]:.3f}) m/rad  (elbow)")
print()

# Compare predicted scatter direction with actual
eps_0 = eps_by_dir[0][0]
sd_x = np.std(eps_0[:,0]) * 100
sd_y = np.std(eps_0[:,1]) * 100
print(f"Actual scatter: SD_x = {sd_x:.3f} cm, SD_y = {sd_y:.3f} cm, ratio = {sd_x/sd_y:.1f}")
# Print your prediction and comparison below
print("[Your prediction and comparison here]")

---## Q3: Intermittent Feedback Corrections (25 points)Build a simple intermittent feedback controller: a noisy primary reach (T = 500 ms), then after a 150 ms sensory delay, sense the hand position and plan a corrective min-jerk to the target (150 ms, also noisy).

### Q3.1 (10 pts) — Implement the corrective submovementThe scaffold below handles the primary movement and the sensory delay coast. Fill in the corrective movement: compute ID torques for a min-jerk from the current hand position to the target, then execute with noise.

In [ ]:
def run_with_corrections(n_corr, rng, T_prim=0.5, sense_delay=0.15, corr_dur=0.15, damping=1.0):
    TGT = start_hand + np.array([0.10, 0.0])
    
    # Precompute primary torques
    t_p, pos_p = minjerk_reach(TGT, T_prim, dt)
    q_p, qd_p, _, tau_p = id_torques(t_p, pos_p, dt)
    
    # Phase 1: primary movement (noisy)
    q = q_p[0].copy(); qd = qd_p[0].copy()
    hand_hist = [fk(q).copy()]
    for i in range(len(tau_p) - 1):
        tau_noisy = add_sdn(tau_p[i], k_noise, rng)
        qdd = joint_accelerations(q, qd, tau_noisy)
        q = q + qd*dt + 0.5*qdd*dt**2; qd = qd + qdd*dt
        hand_hist.append(fk(q).copy())
    
    # Corrections
    for c in range(n_corr):
        # Sensory delay: coast with damping
        for _ in range(int(sense_delay / dt)):
            qdd = joint_accelerations(q, qd, np.zeros(2), B=damping)
            q = q + qd*dt + 0.5*qdd*dt**2; qd = qd + qdd*dt
            hand_hist.append(fk(q).copy())
        
        # Sense current position and compute error
        current_hand = np.array(fk(q))
        error = TGT - current_hand
        if np.linalg.norm(error) < 1e-5: continue
        
        ### YOUR CODE ### (2 lines: plan corrective min-jerk from current_hand to TGT, get ID torques)
        ...  # Replace this line

        # Execute correction with noise
        for j in range(len(tau_c) - 1):
            tau_noisy = add_sdn(tau_c[j], k_noise, rng)
            qdd = joint_accelerations(q, qd, tau_noisy, B=damping*0.5)
            q = q + qd*dt + 0.5*qdd*dt**2; qd = qd + qdd*dt
            hand_hist.append(fk(q).copy())
    
    # Final coast
    for _ in range(int(0.2 / dt)):
        qdd = joint_accelerations(q, qd, np.zeros(2), B=damping)
        q = q + qd*dt + 0.5*qdd*dt**2; qd = qd + qdd*dt
        hand_hist.append(fk(q).copy())
    
    return np.array(hand_hist)

# Quick test
rng_test = np.random.default_rng(42)
h_test = run_with_corrections(1, rng_test)
print(f"Test: {len(h_test)} timesteps, final hand = ({h_test[-1,0]*100:.2f}, {h_test[-1,1]*100:.2f}) cm")

### Q3.2 (10 pts) — Reproduce Figure 8Run 300 trials for 0, 1, and 2 corrections. Plot the velocity profile, endpoint scatter, and bar chart.

In [ ]:
TGT = start_hand + np.array([0.10, 0.0])
tx, ty = TGT[0]*100, TGT[1]*100
N_hw = 300

# Collect endpoints
eps_hw = {}
for nc in [0, 1, 2]:
    rng = np.random.default_rng(77)
    eps = np.zeros((N_hw, 2))
    for trial in range(N_hw):
        h = run_with_corrections(nc, rng)
        eps[trial] = h[-1]
    eps_hw[nc] = eps
    sd = np.std(np.linalg.norm(eps - TGT, axis=1)) * 100
    print(f"  {nc} corrections: SD = {sd:.3f} cm")

sds_hw = [np.std(np.linalg.norm(eps_hw[nc] - TGT, axis=1))*100 for nc in [0,1,2]]

# Single trial for velocity profile
rng_v0 = np.random.default_rng(42); h0 = run_with_corrections(0, rng_v0)
rng_v1 = np.random.default_rng(42); h1 = run_with_corrections(1, rng_v1)

def hand_speed(h):
    dx = np.diff(h[:,0])/dt; dy = np.diff(h[:,1])/dt
    return np.sqrt(dx**2 + dy**2)

spd0 = hand_speed(h0); t0 = np.arange(len(spd0))*dt*1000
spd1 = hand_speed(h1); t1 = np.arange(len(spd1))*dt*1000

# Plot
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = [RED, NAVY, GREEN]

# A: velocity profile
ax = axes[0]
ax.plot(t1, spd1*100, color=NAVY, lw=2.5, label='With 1 correction', zorder=5)
ax.plot(t0, spd0*100, color=RED, lw=2, ls='--', label='No correction', zorder=6)
ax.axvline(500, color=GRAY, ls=':', lw=1.2); ax.axvline(650, color=GREEN, ls=':', lw=1.2)
ax.set_xlabel('Time (ms)'); ax.set_ylabel('Hand speed (cm/s)')
ax.set_title('A. Velocity profile'); ax.legend(fontsize=9)

# B: scatter
ax = axes[1]
for nc, col in zip([0,1,2], colors):
    sd = sds_hw[nc]
    ax.scatter(eps_hw[nc][:,0]*100, eps_hw[nc][:,1]*100, s=6, c=col, alpha=0.3,
               label=f'{nc} corr (SD={sd:.2f})')
circ = plt.Circle((tx,ty), 0.3, fill=False, color='k', lw=1.5, zorder=8)
ax.add_patch(circ); ax.plot(tx, ty, 'k+', ms=10, mew=1.5, zorder=9)
ax.set_xlabel('x (cm)'); ax.set_ylabel('y (cm)'); ax.set_title('B. Endpoint scatter')
ax.legend(fontsize=8); ax.set_aspect('equal')

# C: bar chart
ax = axes[2]
bars = ax.bar([0,1,2], sds_hw, color=colors, width=0.6, edgecolor='white', linewidth=1.5)
for bar, sd in zip(bars, sds_hw):
    ax.text(bar.get_x()+bar.get_width()/2, sd+0.002, f'{sd:.3f}', ha='center', fontsize=10, fontweight='bold')
ax.set_xlabel('Corrections'); ax.set_ylabel('Endpoint SD (cm)'); ax.set_title('C. Diminishing returns')
ax.set_xticks([0,1,2]); ax.set_ylim(0, max(sds_hw)*1.25)
plt.tight_layout(); plt.show()

### Q3.3 (5 pts) — Theoretical minimum with unlimited correctionsCan unlimited corrections drive the endpoint SD to zero? Why or why not? Answer in 2–3 sentences.**Your answer:***[Your answer here]*

---## Q4: Noise Coefficient Sensitivity (15 points)

### Q4.1 (5 pts) — Analytical predictionIf you double k from 0.15 to 0.30, by what factor should endpoint SD change? Derive from the SDN model: σ(t) = k · |τ(t)|.**Your answer:***[Your answer here]*

### Q4.2 (5 pts) — Verify by simulationRun the 6 cm arc at k = 0.15 and k = 0.30. Report both SDs and the ratio.

In [ ]:
t_arc, pos_arc = minjerk_arc(R_ARC, T_ARC, dt)
q_arc, qd_arc, _, tau_arc = id_torques(t_arc, pos_arc, dt)
tgt_arc = pos_arc[-1]
N = 200

sds_k = {}
for k_val in [0.15, 0.30]:
    rng = np.random.default_rng(42)
    eps = np.zeros((N, 2))
    for trial in range(N):
        ### YOUR CODE ### (1 line: call forward_sim_noisy with k_val and store endpoint in eps[trial])
        ...  # Replace this line
    sds_k[k_val] = np.std(np.linalg.norm(eps - tgt_arc, axis=1)) * 100

print(f"k = 0.15: SD = {sds_k[0.15]:.3f} cm")
print(f"k = 0.30: SD = {sds_k[0.30]:.3f} cm")
print(f"Ratio: {sds_k[0.30] / sds_k[0.15]:.2f}x (expected: 2.0x)")

### Q4.3 (5 pts) — Saccadic eye movementsHarris & Wolpert (1998) used k ≈ 0.5 for saccadic eye movements. If you ran your arm simulation at k = 0.5, what endpoint SD would you predict for the 800 ms arc? Would this be physiologically reasonable for arm movements?**Your answer:***[Your answer here]*